### Assignment 7: Modelling and Tracking Shared E-Scooters under Correlated Noise in Urban Environments

##### Group 6: Pavel Kuznetsov, Olga Tiupina, Sofya Konstantinova, Skoltech, 2025

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def generate_true_trajectory(N, T, lambd, sigma_a, x0, v0, a0):
    phi = np.exp(-lambd * T)
    q = sigma_a**2 * (1 - np.exp(-2 * lambd * T))

    x = np.zeros(N)
    v = np.zeros(N)
    a = np.zeros(N)
    x[0], v[0], a[0] = x0, v0, a0

    for i in range(1, N):
        zeta = np.random.normal(0, np.sqrt(q))
        a[i] = phi * a[i - 1] + zeta
        v[i] = v[i - 1] + a[i - 1] * T
        x[i] = x[i - 1] + v[i - 1] * T + 0.5 * a[i - 1] * T**2

    return x, v, a


def generate_measurements(x_true, lambd, sigma_eta, T=1):
    N = len(x_true)
    eta = np.zeros(N)
    
    if lambd >= 1000:
        eta = np.random.normal(0, sigma_eta, N)
    else:
        sigma_zeta = sigma_eta * np.sqrt(1 - np.exp(-2 * lambd * T))
        eta[0] = np.random.normal(0, sigma_eta)
        for i in range(1, N):
            zeta = np.random.normal(0, sigma_zeta)
            eta[i] = np.exp(-lambd * T) * eta[i-1] + zeta
    
    z = x_true + eta
    return z


def build_standard_kalman_matrices(T, sigma_a, sigma_eta):
    Phi = np.array([[1.0, T], [0.0, 1.0]], dtype=float)
    G = np.array([[0.5 * T**2], [T]], dtype=float)
    H = np.array([[1.0, 0.0]], dtype=float)
    Q = sigma_a**2 * (G @ G.T)
    R = np.array([[sigma_eta**2]], dtype=float)
    return Phi, G, H, Q, R


def build_kalman_with_acceleration_matrices(T, lambda_a, sigma_a, sigma_eta):
    phi = np.exp(-lambda_a * T)
    q_zeta = sigma_a**2 * (1.0 - np.exp(-2*lambda_a*T))

    Phi = np.array([[1.0, T, 0.5*T**2], [0.0, 1.0, T], [0.0, 0.0, phi]], dtype=float)
    G = np.array([[0.0], [0.0], [1.0]], dtype=float)
    Q = G @ np.array([[q_zeta]]) @ G.T
    H = np.array([[1.0, 0.0, 0.0]], dtype=float)
    R = np.array([[sigma_eta**2]], dtype=float)
    return Phi, G, H, Q, R


def run_standard_kalman(z, T, sigma_a, sigma_eta, x0_hat, v0_hat, P0):
    Phi, G, H, Q, R = build_standard_kalman_matrices(T, sigma_a, sigma_eta)
    I = np.eye(2)
    n = len(z)
    
    # Storage
    x_pred = np.zeros((n, 2))				# X_{i|i-1}
    P_pred = np.zeros((n, 2, 2))		# P_{i|i-1}
    x_filt = np.zeros((n, 2))				# X_{i|i}
    P_filt = np.zeros((n, 2, 2))		# P_{i|i}
    K_hist = np.zeros((n, 2, 1))		# K_i
    residual = np.zeros(n)					# residual_i
    S_hist = np.zeros(n)						# S_i
    
    # Init
    x_prev = np.array([x0_hat, v0_hat], dtype=float)
    P_prev = P0

    # Loop
    for i in range(n):
        # Predict
        x_pr = Phi @ x_prev									# X_{i|i-1}
        P_pr = Phi @ P_prev @ Phi.T + Q			# P_{i|i-1}

        # Filtration
        resid = z[i] - (H @ x_pr).item()		# residual
        S = (H @ P_pr @ H.T + R).item()			# innovation covariance
        K = P_pr @ H.T / S									# Kalman gain

        x_up = x_pr + K.flatten() * resid		# X_{i|i}
        P_up = (I - K @ H) @ P_pr						# P_{i|i}

        # Save
        x_pred[i], P_pred[i] = x_pr, P_pr
        x_filt[i], P_filt[i] = x_up, P_up
        K_hist[i] = K.reshape(2, 1)
        residual[i] = resid
        S_hist[i] = S

        x_prev, P_prev = x_up, P_up
    
		
		# Standard deviations for analysis
    sigma_x_pred = np.sqrt(P_pred[:, 0, 0])
    sigma_x_filt = np.sqrt(P_filt[:, 0, 0])

    return {
        "Phi": Phi, "G": G, "H": H, "Q": Q, "R": R,
        "x_pred": x_pred, "P_pred": P_pred,
        "x_filt": x_filt, "P_filt": P_filt,
        "K": K_hist, "residual": residual, "S": S_hist,
        "sigma_x_pred": sigma_x_pred, "sigma_x_filt": sigma_x_filt
    }


def run_kalman_with_acceleration(z, T, lambda_a, sigma_a, sigma_eta, x0_hat, v0_hat, a0_hat, P0):
    Phi, G, H, Q, R = build_kalman_with_acceleration_matrices(T, lambda_a, sigma_a, sigma_eta)
    I = np.eye(3)
    n = len(z)

    # Storage
    x_pred = np.zeros((n, 3))				# X_{i|i-1} [x, v, a]
    P_pred = np.zeros((n, 3, 3))		# P_{i|i-1}
    x_filt = np.zeros((n, 3))				# X_{i|i}
    P_filt = np.zeros((n, 3, 3))		# P_{i|i}
    K_hist = np.zeros((n, 3, 1))		# K_i
    residual = np.zeros(n)					# residual_i
    S_hist = np.zeros(n)						# S_i

    # Init
    x_prev = np.array([x0_hat, v0_hat, a0_hat], dtype=float)
    P_prev = P0.astype(float)
    
		# Loop
    for i in range(n):
        # Predict
        x_pr = Phi @ x_prev
        P_pr = Phi @ P_prev @ Phi.T + Q

        # Filtration
        resid = z[i] - (H @ x_pr).item()		# residual
        S = (H @ P_pr @ H.T + R).item()			# innovation covariance
        K = P_pr @ H.T / S									# Kalman gain
        
        x_up = x_pr + K.flatten() * resid
        P_up = (I - K @ H) @ P_pr

        # Save results
        x_pred[i], P_pred[i] = x_pr, P_pr
        x_filt[i], P_filt[i] = x_up, P_up
        K_hist[i] = K.reshape(3, 1)
        residual[i] = resid
        S_hist[i] = S

        x_prev, P_prev = x_up, P_up

    # Standard deviations for analysis
    sigma_x_pred = np.sqrt(P_pred[:, 0, 0])
    sigma_x_filt = np.sqrt(P_filt[:, 0, 0])
    sigma_v_filt = np.sqrt(P_filt[:, 1, 1])
    sigma_a_filt = np.sqrt(P_filt[:, 2, 2])

    return {
        "Phi": Phi, "G": G, "H": H, "Q": Q, "R": R,
        "x_pred": x_pred, "P_pred": P_pred,
        "x_filt": x_filt, "P_filt": P_filt,
        "K": K_hist, "residual": residual, "S": S_hist,
        "sigma_x_pred": sigma_x_pred, "sigma_x_filt": sigma_x_filt,
        "sigma_v_filt": sigma_v_filt, "sigma_a_filt": sigma_a_filt
    }


def calculate_residual_variance(P_predicted, H, R):
    num_points = len(P_predicted)
    S = np.zeros(num_points)
    for i in range(num_points):
        S_value = (H @ P_predicted[i] @ H.T + R)[0, 0]
        S[i] = S_value
    return S


def run_standard_kalman_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a, lambda_eta):
    errors = np.zeros((M, N))
    sigmas = np.zeros((M, N))

    for m in range(M):
        x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_a, sigma_a, x0, v0, a0)
        z = generate_measurements(x_true, lambda_eta, sigma_eta)
        
        res = run_standard_kalman(z, T, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, P0=np.diag([10000, 10000]))
        
        errors[m, :] = res["x_filt"][:, 0] - x_true
        sigmas[m, :] = res["sigma_x_filt"]

    rms_true = np.sqrt(np.mean(errors**2, axis=0))
    mean_sigma = np.mean(sigmas, axis=0)
    return rms_true, mean_sigma


def run_kalman_with_acceleration_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a, lambda_eta):
    errors_fx = np.zeros((M, N))
    errors_fv = np.zeros((M, N))
    errors_fa = np.zeros((M, N))
    sigmas_fx = np.zeros((M, N))
    sigmas_fv = np.zeros((M, N))
    sigmas_fa = np.zeros((M, N))
    
    errors_px = np.zeros((M, N))
    errors_pv = np.zeros((M, N))
    errors_pa = np.zeros((M, N))
    sigmas_px = np.zeros((M, N))
    sigmas_pv = np.zeros((M, N))
    sigmas_pa = np.zeros((M, N))

    for m in range(M):
        x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_a, sigma_a, x0, v0, a0)
        z = generate_measurements(x_true, lambda_eta, sigma_eta)
        
        res = run_kalman_with_acceleration(z, T, lambda_a, sigma_a, sigma_eta, x0_hat=2, v0_hat=0, a0_hat=0, P0=np.diag([10000, 10000, 10000]))
        
        errors_fx[m, :] = res["x_filt"][:, 0] - x_true
        errors_fv[m, :] = res["x_filt"][:, 1] - v_true
        errors_fa[m, :] = res["x_filt"][:, 2] - a_true
        sigmas_fx[m, :] = res["sigma_x_filt"]
        sigmas_fv[m, :] = res["sigma_v_filt"]
        sigmas_fa[m, :] = res["sigma_a_filt"]
        
        errors_px[m, :] = res["x_pred"][:, 0] - x_true
        errors_pv[m, :] = res["x_pred"][:, 1] - v_true
        errors_pa[m, :] = res["x_pred"][:, 2] - a_true
        sigmas_px[m, :] = np.sqrt(res["P_pred"][:, 0, 0])
        sigmas_pv[m, :] = np.sqrt(res["P_pred"][:, 1, 1])
        sigmas_pa[m, :] = np.sqrt(res["P_pred"][:, 2, 2])

    rmse_fx = np.sqrt(np.mean(errors_fx**2, axis=0))
    rmse_fv = np.sqrt(np.mean(errors_fv**2, axis=0))
    rmse_fa = np.sqrt(np.mean(errors_fa**2, axis=0))
    
    rmse_px = np.sqrt(np.mean(errors_px**2, axis=0))
    rmse_pv = np.sqrt(np.mean(errors_pv**2, axis=0))
    rmse_pa = np.sqrt(np.mean(errors_pa**2, axis=0))
    
    mean_sigma_fx = np.mean(sigmas_fx, axis=0)
    mean_sigma_fv = np.mean(sigmas_fv, axis=0)
    mean_sigma_fa = np.mean(sigmas_fa, axis=0)
    
    mean_sigma_px = np.mean(sigmas_px, axis=0)
    mean_sigma_pv = np.mean(sigmas_pv, axis=0)
    mean_sigma_pa = np.mean(sigmas_pa, axis=0)
    
    return (rmse_fx, rmse_fv, rmse_fa, rmse_px, rmse_pv, rmse_pa, mean_sigma_fx, mean_sigma_fv, mean_sigma_fa, mean_sigma_px, mean_sigma_pv, mean_sigma_pa)

In [ ]:
fontsize_title = 30
fontsize_label = 20
fontsize_leg = 20

#### Part I. Effects of Ignoring Correlated State and Measurement Noise in Kalman Filtering Background and useful recommendations

In [ ]:
# Parametrs
N, M = 200, 500
T = 1.0
sigma_a, sigma_eta = 0.2, 20.0
x0, v0, a0 = 5.0, 1.0, 0.0
lambda_corr, lambda_uncorr = 0.1, 1000

P0_standard = np.diag([10000, 10000])
P0_with_acceleration = np.diag([10000, 10000, 10000])

t = np.arange(N) * T

In [ ]:
# Generate true trajectory with different lambda
x_true_1, v_true_1, a_true_1 = generate_true_trajectory(N, T, lambda_corr, sigma_a, x0, v0, a0)
x_true_2, v_true_2, a_true_2 = generate_true_trajectory(N, T, lambda_uncorr, sigma_a, x0, v0, a0)

In [ ]:
# Plotting correlated and uncorrelated acceleration
plt.figure(figsize=(25, 8))

plt.plot(t, a_true_1, color='blue', linewidth=1.5, alpha=1.0, label="Correlated acceleration (λ = 0.1)")
plt.plot(t, a_true_2, color='orange', linewidth=1.5, alpha=1.0, label="Uncorrelated acceleration (λ = 1000)")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Acceleration, m/s² \n", fontsize=fontsize_label)
plt.title("\n Fig.1. True acceleration for different λ values \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

Fig.1. illustrates the difference between correlated and uncorrelated acceleration:

- When λ = 0.1, the acceleration changes smoothly over time, showing a strong correlation between successive values (correlated acceleration)
- When λ = 1000, it fluctuates randomly, representing uncorrelated white noise (uncorrelated acceleration)

The graph confirms that a smaller λ value results in smoother and more stable acceleration behavior.

#### Scenario 1 — Optimal Kalman filter for uncorrelated state and measurement noise

In [ ]:
# Generate trajectory and measurements
x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_uncorr, sigma_a, x0, v0, a0)
z = generate_measurements(x_true, lambda_uncorr, sigma_eta)

# Run standard Kalman filter
res = run_standard_kalman(z, T, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, P0=P0_standard)

In [ ]:
# Run multiple simulations for statistical analysis
rms_true, mean_sigma = run_standard_kalman_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a=lambda_uncorr, lambda_eta=lambda_uncorr)

In [ ]:
# Plotting position estimates and measurements
plt.figure(figsize=(20, 8))

plt.plot(t, x_true, color='blue', linewidth=1.5, alpha=0.8, label="True position")
plt.plot(t, z, color='black', linewidth=1.0, alpha=0.8, label="Measurements")
plt.plot(t, res["x_filt"][:, 0], color='orange', linewidth=1.5, alpha=1.0, label="Filtered estimate")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Position, m \n", fontsize=fontsize_label)
plt.title("\n Fig.2. Scenario 1 — Uncorrelated state and uncorrelated measurement noise \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

# Plotting RMSE comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.3. Scenario 1 — Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

# Plotting filter performance comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true, color='blue', linewidth=1.5, alpha=0.8, label="Filtered positions (Final error KF)")
plt.axhline(sigma_eta, color="red", linestyle="--", linewidth=2.0, alpha=1.0, label="Measurement noise σ_η")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.4. Scenario 1 — Comparison of final error for filtered estimate \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

- Fig.2 demonstrates that the Kalman filter provides accurate trajectory estimation under uncorrelated noise conditions, with the filtered estimate closely following the true position while effectively smoothing measurement noise.

- Fig.3 shows excellent alignment between the true estimation error and calculated estimation error across 500 filter runs, confirming the filter's statistical consistency and optimal performance.

- Fig.4 proves that the filter significantly improves measurement accuracy - the final error of filtered estimates is substantially lower than the original measurement noise σ_η.

These results validate that the Kalman filter constructed under accurate assumptions about uncorrelated state and measurement noise provides optimal estimates, allowing us to confidently rely on the calculated error for decision-making.

#### Scenario 2 - Effects of ignoring correlated state noise in Kalman filtering. Non-optimal filter: assumes uncorrelated state noise, but true acceleration is correlated

In [ ]:
# Generate trajectory with correlated state noise and uncorrelated measurements
x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_corr, sigma_a, x0, v0, a0)
z = generate_measurements(x_true, lambda_uncorr, sigma_eta)

# Run standard Kalman filter
res = run_standard_kalman(z, T, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, P0=P0_standard)

In [ ]:
# Run multiple simulations for statistical analysis - Scenario 2
rms_true, mean_sigma = run_standard_kalman_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a=lambda_corr, lambda_eta=lambda_uncorr)

In [ ]:
# Plotting position estimates and measurements
plt.figure(figsize=(20, 8))

plt.plot(t, x_true, color='blue', linewidth=1.5, alpha=0.8, label="True position")
plt.plot(t, z, color='black', linewidth=1.0, alpha=0.8, label="Measurements")
plt.plot(t, res["x_filt"][:, 0], color='orange', linewidth=1.5, alpha=1.0, label="Filtered estimate")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Position, m \n", fontsize=fontsize_label)
plt.title("\n Fig.5. Scenario 2 — Correlated acceleration and uncorrelated measurement noise \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting RMSE comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.6. Scenario 2 — Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting filter performance comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true, color='blue', linewidth=1.5, alpha=0.8, label="Filtered positions (Final error KF)")
plt.axhline(sigma_eta, color="red", linestyle="--", linewidth=2.0, alpha=1.0, label="Measurement noise σ_η")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.7. Scenario 2 — Comparison of final error for filtered estimate \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

- Fig.5 demonstrates trajectory estimation when correlated state noise is neglected in the filter design, showing visible deviations between filtered estimates and true position.

- Fig.6 reveals that the true estimation error significantly exceeds the calculated estimation error across 500 runs, confirming the filter's non-optimal performance when correlated acceleration (λ=0.1) is ignored.

- Fig.7 shows that despite some improvement over raw measurements, the filter's performance is degraded compared to Scenario 1, and we cannot rely on the calculated error for decision-making due to the incorrect noise assumptions.

These results validate that neglecting correlated state noise leads to non-optimal filter performance and unreliable error estimation.

#### Scenario 3 - Impact of correlated measurement noise on filter consistency. Non-optimal filter: assumes uncorrelated measurement noise, but true observations are correlated

In [ ]:
# Generate trajectory with white acceleration and correlated measurements
x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_uncorr, sigma_a, x0, v0, a0)
z_corr = generate_measurements(x_true, lambda_corr, sigma_eta)  # коррелированные измерения

# Run standard Kalman filter (ignoring measurement correlation)
res3 = run_standard_kalman(z_corr, T, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, P0=P0_standard)

In [ ]:
# Run multiple simulations for statistical analysis
rms_true_3, mean_sigma_3 = run_standard_kalman_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a=lambda_uncorr, lambda_eta=lambda_corr)

In [ ]:
# Plotting position estimates and measurements
plt.figure(figsize=(20, 8))

plt.plot(t, x_true, color='blue', linewidth=1.5, alpha=0.8, label="True position")
plt.plot(t, z_corr, color='black', linewidth=1.0, alpha=0.8, label="Measurements")
plt.plot(t, res3["x_filt"][:, 0], color='orange', linewidth=1.5, alpha=1.0, label="Filtered estimate")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Position, m \n", fontsize=fontsize_label)
plt.title("\n Fig.8. Scenario 3 — Uncorrelated acceleration and correlated measurement noise \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting RMSE comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true_3, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma_3, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.9. Scenario 3 — Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting filter performance comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true_3, color='blue', linewidth=1.5, alpha=0.8, label="Filtered positions (Final error KF)")
plt.axhline(sigma_eta, linestyle="--", color="red", linewidth=2.0, alpha=1.0, label="Measurement noise σ_η")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.10. Scenario 3 — Comparison of final error for filtered estimate \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

- Fig.8 demonstrates trajectory estimation when correlated measurement noise is neglected in the filter design, showing the challenge of filtering correlated measurements.

- Fig.9 reveals that the true estimation error exceeds the calculated estimation error across 500 runs, confirming non-optimal performance when correlated measurement noise (λ_η=0.1) is ignored.

- Fig.10 shows that the filter provides some improvement over raw correlated measurements, though the performance is degraded compared to the optimal case.

Comparison with Scenario 2 indicates that neglecting correlated state noise results in greater accuracy degradation than neglecting correlated measurement noise under these noise statistics, as state noise correlation more severely impacts the system dynamics model.

#### Scenario 4. Combined effects of ignoring correlated state and measurement noise. Non- optimal Kalman filter: assumed uncorrelated state and measurement noise vs. actual correlation of both noises

In [ ]:
# Generate trajectory with correlated acceleration and correlated measurements
x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_corr, sigma_a, x0, v0, a0)
z_corr = generate_measurements(x_true, lambda_corr, sigma_eta)

# Run standard Kalman filter (ignoring both correlations)
res4 = run_standard_kalman(z_corr, T, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, P0=P0_standard)

In [ ]:
# Run multiple simulations for statistical analysis - Scenario 4
rms_true_4, mean_sigma_4 = run_standard_kalman_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a=lambda_corr, lambda_eta=lambda_corr)

In [ ]:
# Plotting position estimates and measurements
plt.figure(figsize=(20, 8))

plt.plot(t, x_true, color='blue', linewidth=1.5, alpha=0.8, label="True position")
plt.plot(t, z_corr, color='black', linewidth=1.0, alpha=0.8, label="Measurements")
plt.plot(t, res4["x_filt"][:, 0], color='orange', linewidth=1.5, alpha=1.0, label="Filtered estimate")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Position, m \n", fontsize=fontsize_label)
plt.title("\n Fig.11. Scenario 4 — Correlated acceleration and correlated measurement noise \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting RMSE comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true_4, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma_4, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.12. Scenario 4 — Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting filter performance comparison
plt.figure(figsize=(20, 8))

plt.plot(t, rms_true_4, color='blue', linewidth=1.5, alpha=0.8, label="Filtered positions (Final error KF)")
plt.axhline(sigma_eta, linestyle="--", color="red", linewidth=2.0, alpha=1.0, label="Measurement noise σ_η")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.13. Scenario 4 — Comparison of final error for filtered estimate \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

- Fig.11 demonstrates the challenging scenario where both state and measurement noises are correlated but ignored in the filter design.

- Fig.12 shows the most severe discrepancy between true and calculated estimation errors across all scenarios, confirming the highest degree of non-optimality when both noise correlations are neglected.

- Fig.13 reveals that while the filter still provides marginal improvement over raw measurements, its usefulness is significantly limited compared to previous scenarios.

Comparison with Scenarios 2 and 3 demonstrates that the combined neglect of both noise correlations results in the worst filter performance, with the true estimation error showing the largest deviation from optimal behavior and the calculated uncertainty being the most unreliable for decision-making.

### Part II. Optimal Kalman Filtering with Correlated State Noise

In [ ]:
# Generate trajectory with correlated acceleration and white measurements
x_true, v_true, a_true = generate_true_trajectory(N, T, lambda_corr, sigma_a, x0, v0, a0)
z = generate_measurements(x_true, lambda_uncorr, sigma_eta)

# Run extended Kalman filter for correlated state noise
res = run_kalman_with_acceleration(z, T, lambda_corr, sigma_a, sigma_eta, x0_hat=0, v0_hat=0, a0_hat=0, P0=P0_with_acceleration)

In [ ]:
# Plotting position estimates and measurements
plt.figure(figsize=(20, 8))

plt.plot(t, x_true, color='blue', linewidth=1.5, alpha=0.8, label="True position")
plt.plot(t, z, color='black', linewidth=1.0, alpha=0.8, label="Measurements")
plt.plot(t, res["x_filt"][:, 0], color='orange', linewidth=1.5, alpha=1.0, label="Filtered estimate")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Position, m \n", fontsize=fontsize_label)
plt.title("\n Fig.14. Optimal Kalman Filter with Gauss–Markov acceleration \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

Fig.14 demonstrates the performance of the optimal Kalman filter that properly accounts for correlated acceleration using Gauss-Markov modeling. The filtered estimate accurately tracks the true trajectory while effectively smoothing measurement noise, showing significant improvement over previous non-optimal scenarios. This confirms that when noise correlations are correctly modeled in the filter design, the Kalman filter provides optimal estimates with reliable performance even under correlated noise conditions.

In [ ]:
# Run multiple simulations for extended Kalman filter - ALL estimates
(rmse_fx, rmse_fv, rmse_fa, 
 rmse_px, rmse_pv, rmse_pa,
 mean_sigma_fx, mean_sigma_fv, mean_sigma_fa,
 mean_sigma_px, mean_sigma_pv, mean_sigma_pa) = run_kalman_with_acceleration_multiple(M, N, T, sigma_a, sigma_eta, x0, v0, a0, lambda_a=lambda_corr, lambda_eta=lambda_uncorr)

In [ ]:
# Plotting filtered position errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_fx, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma_fx, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.15. Filtered Position: Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting filtered velocity errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_fv, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma_fv, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m/s \n", fontsize=fontsize_label)
plt.title("\n Fig.16. Filtered Velocity: Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting filtered acceleration errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_fa, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for filtered estimate")
plt.plot(t, mean_sigma_fa, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m/s² \n", fontsize=fontsize_label)
plt.title("\n Fig.17. Filtered Acceleration: Filtration errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

Fig.15, Fig.16 and Fig.17 demonstrate the optimal performance of the extended Kalman filter that accounts for correlated acceleration. The excellent alignment between true estimation errors and calculated estimation errors across all three state variables (position, velocity, and acceleration) confirms the filter's statistical consistency and optimality. This validates that the extended state vector approach successfully handles correlated noise dynamics, providing reliable uncertainty quantification for all estimated parameters and enabling confident reliance on the calculated errors for decision-making.

In [ ]:
# Plotting predicted position errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_px, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for predicted estimate")
plt.plot(t, mean_sigma_px, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m \n", fontsize=fontsize_label)
plt.title("\n Fig.18. Predicted Position: Prediction errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting predicted velocity errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_pv, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for predicted estimate")
plt.plot(t, mean_sigma_pv, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m/s \n", fontsize=fontsize_label)
plt.title("\n Fig.19. Predicted Velocity: Prediction errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()


# Plotting predicted acceleration errors
plt.figure(figsize=(20, 8))

plt.plot(t, rmse_pa, color='blue', linewidth=1.5, alpha=0.8, label="True estimation error for predicted estimate")
plt.plot(t, mean_sigma_pa, color='orange', linewidth=1.5, alpha=1.0, label="Calculated estimation error")

plt.xlabel("\n Time, s \n", fontsize=fontsize_label)
plt.ylabel("\n Error, m/s² \n", fontsize=fontsize_label)
plt.title("\n Fig.20. Predicted Acceleration: Prediction errors \n", fontsize=fontsize_title)
plt.legend(fontsize=fontsize_leg)
plt.grid(True)
plt.show()

Fig.18, Fig.19 and Fig.20 demonstrate the prediction performance of the optimal Kalman filter with Gauss-Markov acceleration modeling. The close alignment between true estimation errors and calculated estimation errors for all predicted state variables (position, velocity, and acceleration) confirms the filter's consistent performance in prediction mode. This validates that the state-space model properly accounts for correlated acceleration dynamics, providing reliable uncertainty quantification not only for filtered estimates but also for predicted estimates, ensuring the filter's reliability for forecasting applications under correlated noise conditions.

#### Conclusion

During this assignment, we:

- Developed and tested Kalman filters under various noise correlation scenarios, systematically analyzing the impact of correct and incorrect noise assumptions on filter performance and estimation accuracy.

- Implemented Scenario 1 (optimal filter) with uncorrelated state and measurement noises, demonstrating that the Kalman filter provides optimal estimates with reliable uncertainty quantification when noise assumptions match reality. The true estimation errors aligned closely with calculated errors across 500 runs, validating the filter's statistical consistency.

- Investigated non-optimal scenarios (2-4) where the filter incorrectly assumed uncorrelated noises while the true system had correlated components. Results showed significant performance degradation:

	- Scenario 2 (correlated state noise): True estimation errors significantly exceeded calculated errors, indicating unreliable uncertainty quantification when state noise correlation is neglected

	- Scenario 3 (correlated measurement noise): Similar degradation occurred, with comparative analysis revealing that neglecting state noise correlation causes greater accuracy reduction than neglecting measurement noise correlation

	- Scenario 4 (both noises correlated): Showed the most severe performance deterioration, confirming that multiple incorrect assumptions compound filter non-optimality

- Designed an optimal Kalman filter for correlated state noise by extending the state vector to include acceleration dynamics. This approach successfully handled correlated acceleration, providing accurate estimates and reliable uncertainty quantification for position, velocity, and acceleration in both filtered and predicted modes.

- Verified statistical consistency through extensive simulations (M=500 runs), demonstrating close alignment between true and calculated estimation errors for the optimal filter across all state variables.

The assignment demonstrates that proper noise modeling is crucial for Kalman filter performance. When noise correlations match filter assumptions, the filter provides optimal estimates with reliable uncertainty bounds. However, neglecting noise correlations leads to significant performance degradation and unreliable error estimates, with state noise neglect having greater impact than measurement noise neglect under the tested conditions.